# PC 및 주변기기 판매 쇼핑몰의 상품 추천 시스템

## 추천 시스템의 협업 필터링이란?

실제 쇼핑몰에서 활용할 수 있는 협업 필터링(Collaborative Filtering) 방식 중 하나인 사용자 행동 기반의 아이템-아이템 협업 필터링(Item-Based Collaborative Filtering)을 사용

1. 사용자 기반
2. 아이템 기반
3. 사용자+아이템 기반

| 고객 | CPU | GPU | RAM | SSD | 키보드 |
|-------|-----|-----|-----|-----|-----|
| 고객 A | 5 | 5 | 4 | 5 | 0 |
| 고객 B | 5 | 5 | 0 | 4 | 0 |
| 고객 C | 0 | 0 | 0 | 0 | 5 |
| 고객 D | 5 | 4 | 5 | 5 | 0 |

이 때 고객 A와 고객 D가 가장 선호하는 패턴이 유사하다고 볼 수 있으므로 만약, 고객 A가 파워서플라이를 구매하여 5라는 만족도를 표시했다면,
고객 D에게도 파워서플라이 구매를 추천할 수 있음.

In [4]:
# 1단계: 쇼핑몰 상품 목록 불러오기
import pandas as pd

product_df = pd.read_csv('data/dummy_products.csv', encoding='utf-8-sig')

print('상품 수:', len(product_df))
print('카테고리 수:', product_df['category'].nunique())
print(product_df.head(10))

상품 수: 120
카테고리 수: 12
   product_id category
0           1      GPU
1           2      GPU
2           3      GPU
3           4      GPU
4           5      GPU
5           6      GPU
6           7      GPU
7           8      GPU
8           9      GPU
9          10      GPU


In [ ]:
# 2단계: 고객 구매 이력 불러오기
purchase_df = pd.read_csv('data/dummy_purchase_history.csv', encoding='utf-8-sig')

# 같은 회원이 같은 상품을 여러 번 샀어도 여기서는 '구매 경험 있음(1)'으로 처리합니다.
purchase_df['purchased'] = 1

print('구매 이력 행 수:', len(purchase_df))
print('구매 회원 수:', purchase_df['member_id'].nunique())
print('구매 상품 수:', purchase_df['product_id'].nunique())
print(purchase_df.head(10))

assert purchase_df['product_id'].isin(product_df['product_id']).all(), '상품 목록에 없는 product_id가 있습니다.'

구매 이력 행 수: 13226
구매 회원 수: 1720
구매 상품 수: 120
   order_id  member_id  product_id   category  quantity           ordered_at  \
0         1        611          96        RAM         1  2026-01-01 00:05:18   
1         1        611          82      POWER         1  2026-01-01 00:05:18   
2         1        611         116  MAINBOARD         1  2026-01-01 00:05:18   
3         2       1506           5        GPU         1  2026-01-01 01:43:05   
4         2       1506          88      POWER         1  2026-01-01 01:43:05   
5         2       1506          18       CASE         2  2026-01-01 01:43:05   
6         3        649          23        CPU         1  2026-01-01 02:18:03   
7         4        111          53        SSD         1  2026-01-01 04:59:42   
8         4        111          77        HDD         1  2026-01-01 04:59:42   
9         4        111          95        RAM         1  2026-01-01 04:59:42   

   purchased  
0          1  
1          1  
2          1  
3          1  


In [6]:
# 3단계: 사용자-상품 구매 여부 피벗테이블 생성
user_item_matrix = purchase_df.pivot_table(
    index='member_id',
    columns='product_id',
    values='purchased',
    aggfunc='max',  # 같은 상품을 여러 번 사도 값은 1
    fill_value=0
)

print('행(구매 회원) x 열(상품):', user_item_matrix.shape)
print(user_item_matrix.iloc[:5, :12])

행(구매 회원) x 열(상품): (1720, 120)
product_id  1   2   3   4   5   6   7   8   9   10  11  12
member_id                                                 
1            0   0   1   0   0   0   0   0   0   0   0   0
3            0   0   1   0   0   0   0   0   0   0   0   0
4            0   0   0   0   0   0   0   0   0   0   0   0
6            0   0   0   0   0   0   0   0   0   0   0   0
7            0   0   0   0   0   0   0   0   0   0   0   0


1. 숫자 5: 해당 고객이 해당 상품을 매우 선호함
2. 숫자 4: 해당 고객이 해당 상품을 선호함
3. 숫자 0: 구매한 적이 없거나 평가한 기록이 없음

In [7]:
# 4단계: 아이템-아이템 협업 필터링 모델 학습
# 코사인 유사도(Cosine Similarity): 이 때 가장 패턴이 가까운 정도를 나타내는 점수
# 4.1 상품 간 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

item_user_matrix = user_item_matrix.T
item_similarity = cosine_similarity(item_user_matrix)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

print('상품 유사도 행렬 크기:', item_similarity_df.shape)
print(item_similarity_df.iloc[:6, :6].round(2))

상품 유사도 행렬 크기: (120, 120)
product_id     1     2     3     4     5     6
product_id                                    
1           1.00  0.14  0.14  0.07  0.08  0.08
2           0.14  1.00  0.12  0.11  0.03  0.08
3           0.14  0.12  1.00  0.06  0.09  0.07
4           0.07  0.11  0.06  1.00  0.10  0.07
5           0.08  0.03  0.09  0.10  1.00  0.15
6           0.08  0.08  0.07  0.07  0.15  1.00


In [8]:
# 5단계: 고객별 상품 추천 함수 구현 - 해당 상품의 코사인 유사도가 가장 높은 상품을 추천한다.
def recommend_products(user_id, user_item_matrix, item_similarity_df, product_df, top_n=5):
    if user_id not in user_item_matrix.index:
        print('구매 기록이 없는 신규 회원입니다. 기본 인기 상품 추천은 별도 구현 예정입니다.')
        return pd.DataFrame()

    # 원래 코드는 user_ratings였지만, 지금은 구매 여부(0 또는 1)입니다.
    user_purchases = user_item_matrix.loc[user_id]
    purchased_items = user_purchases[user_purchases > 0]

    scores = pd.Series(0.0, index=item_similarity_df.index)
    for product_id, purchased in purchased_items.items():
        scores += item_similarity_df[product_id] * purchased

    # 이미 구매한 상품은 추천에서 제외
    scores = scores.drop(purchased_items.index, errors='ignore')
    recommendations = scores[scores > 0].sort_values(ascending=False).head(top_n)

    result = pd.DataFrame({
        'product_id': recommendations.index,
        'recommend_score': recommendations.values
    })
    result = result.merge(product_df, on='product_id', how='left')

    # dummy_products.csv에는 name과 price가 없으므로 두 열은 요청하지 않습니다.
    return result[['product_id', 'category', 'recommend_score']]

In [9]:
# 적용: 사용자에게 상품을 추천
user_id = int(user_item_matrix.index[0])
print('테스트 회원 ID:', user_id)

purchased_ids = user_item_matrix.loc[user_id]
purchased_ids = purchased_ids[purchased_ids > 0].index.tolist()
print('이미 구매한 상품 ID:', purchased_ids)

result = recommend_products(
    user_id=user_id,
    user_item_matrix=user_item_matrix,
    item_similarity_df=item_similarity_df,
    product_df=product_df,
    top_n=5
)
print('추천 상품:')
print(result)

테스트 회원 ID: 1
이미 구매한 상품 ID: [3, 22, 23, 33, 37, 42, 52, 101, 103, 107, 111]
추천 상품:
   product_id  category  recommend_score
0          81     POWER         1.799964
1          31  KEYBOARD         1.771719
2          32  KEYBOARD         1.746752
3          21       CPU         1.716098
4          51       SSD         1.654002


In [10]:
# 6단계: 결과 저장 — 원본 실습 item_cf_model.pkl을 덮어쓰지 않도록 새 이름 사용
from pathlib import Path
import joblib

Path('models').mkdir(exist_ok=True)
model = {
    'user_item_matrix': user_item_matrix,
    'item_similarity_df': item_similarity_df,
    'product_df': product_df
}

joblib.dump(model, 'models/item_cf_model_pc120.pkl')
print('저장 완료: models/item_cf_model_pc120.pkl')

저장 완료: models/item_cf_model_pc120.pkl


In [11]:
# 확인용: 기존에 구매 기록이 없는 신규 회원은 빈 데이터프레임이 반환됩니다.
print(recommend_products(999999, user_item_matrix, item_similarity_df, product_df))

구매 기록이 없는 신규 회원입니다. 기본 인기 상품 추천은 별도 구현 예정입니다.
Empty DataFrame
Columns: []
Index: []
